# Libraries and Setup

In [ ]:
try:
    import qiskit
except ImportError:
    !pip install -U qiskit
    !pip install -U qiskit-ibm-runtime
    !pip install -U pylatexenc
    !pip install -U qiskit-aer
    !pip install -U qiskit-algorithms

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
import networkx as nx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorEstimator
from qiskit.primitives import StatevectorSampler
from qiskit.primitives import BackendEstimatorV2, BackendSamplerV2
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

from qiskit_algorithms.optimizers import L_BFGS_B
from qiskit.synthesis import LieTrotter

from functools import partial
import itertools
import time

# Helper Functions

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator, objective_func_vals_list):
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = -results.data.evs

    objective_func_vals_list.append(cost)

    return cost

In [ ]:
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]

In [ ]:
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )

In [ ]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    return sum(
        graph.get_edge_data(u, v) * (x[u] * (1 - x[v]) + x[v] * (1 - x[u]))
        for u, v in list(graph.edge_list())
    )

In [ ]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)

def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_cost = fval
            min_sol = candidate_sol

    return min_sol

In [ ]:
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)

def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)

def samples_to_objective_values(samples, hamiltonian):
    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [ ]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    return sum(
        graph.get_edge_data(u, v) * (x[u] * (1 - x[v]) + x[v] * (1 - x[u]))
        for u, v in list(graph.edge_list())
    )

In [ ]:
def build_max_cut_paulis(graph: rx.PyGraph) -> list[tuple[str, float]]:
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list

In [ ]:
def get_total_edge_weight(graph: rx.PyGraph) -> float:
    total_weight = 0.0
    for u, v in graph.edge_list():
        total_weight += graph.get_edge_data(u, v)
    return total_weight

# QGOA

In [ ]:
def build_qgoa_circuit(graph, reps: int = 1):
    num_qubits = len(graph.nodes())
    circuit = QuantumCircuit(num_qubits)
    circuit.h(range(num_qubits))

    for layer in range(reps):
        for qubit in range(num_qubits):
            theta_y = Parameter(f"theta_y_{layer}_{qubit}")
            theta_z = Parameter(f"theta_z_{layer}_{qubit}")
            circuit.ry(theta_y, qubit)
            circuit.rz(theta_z, qubit)

        eta = Parameter(f"eta_{layer}")
        pauli_list_xx = []
        pauli_list_yy = []

        for edge in graph.edge_list():
            u, v = edge
            weight = graph.get_edge_data(u, v)
            pauli_list_xx.append(("XX", [u, v], weight))
            pauli_list_yy.append(("YY", [u, v], weight))

        pauli_list = pauli_list_xx + pauli_list_yy

        hamiltonian = SparsePauliOp.from_sparse_list(pauli_list, num_qubits=num_qubits)

        synthesis = LieTrotter(reps=2)
        evol_gate = PauliEvolutionGate(hamiltonian, time=eta, synthesis=synthesis)
        circuit.append(evol_gate, range(num_qubits))

    return circuit

In [ ]:
def run_qgoa(graph, num_layers):
  max_cut_paulis = build_max_cut_paulis(graph)

  cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, len(graph))

  circuit = build_qgoa_circuit(graph, reps=num_layers)

  pm = generate_preset_pass_manager(optimization_level=3)

  candidate_circuit = pm.run(circuit)

  num_qubits = len(graph)

  init_params = []
  for param in circuit.parameters:
      if param.name.startswith('eta'):
          init_params.append(np.random.uniform(-0.01, 0.01))
      else:
          init_params.append(np.random.uniform(0, 2 * np.pi))

  objective_func_vals = []

  optimizer = L_BFGS_B(maxiter=100)

  bound_cost_func = partial(cost_func_estimator,
                            ansatz=candidate_circuit,
                            hamiltonian=cost_hamiltonian,
                            estimator=estimator,
                            objective_func_vals_list=objective_func_vals)

  result = optimizer.minimize(
      fun=bound_cost_func,
      x0=init_params
  )

  optimized_circuit = candidate_circuit.assign_parameters(result.x)
  optimized_circuit.measure_all()

  pub = (optimized_circuit,)
  job = sampler.run([pub])

  result = job.result()[0]
  counts_bin = result.data.meas.get_counts()

  counts_int = {int(k, 2): v for k, v in counts_bin.items()}
  shots = sum(counts_int.values())
  final_distribution_int = {
      key: val / shots for key, val in counts_int.items()
  }

  best_sol = best_solution(final_distribution_int, cost_hamiltonian)
  best_sol_bitstring = to_bitstring(int(best_sol), len(graph))
  best_sol_bitstring.reverse()

  cut_value = evaluate_sample(best_sol_bitstring, graph)

  result_dist = samples_to_objective_values(
    final_distribution_int, cost_hamiltonian
  )

  return cut_value, best_sol_bitstring, objective_func_vals

# QAOA

In [ ]:
def build_qaoa_circuit(cost_hamiltonian, reps=1):
    num_qubits = cost_hamiltonian.num_qubits

    gamma_params = [Parameter(f"γ_{i}") for i in range(reps)]
    beta_params = [Parameter(f"β_{i}") for i in range(reps)]

    circuit = QuantumCircuit(num_qubits)

    circuit.h(range(num_qubits))

    for rep in range(reps):
        pauli_list = cost_hamiltonian.to_list()

        for pauli, coeff in pauli_list:
            if 'ZZ' in pauli:
                indices = [i for i, p in enumerate(pauli[::-1]) if p == 'Z']
                if len(indices) == 2:
                    i, j = indices
                    circuit.rzz(2 * coeff * gamma_params[rep], i, j)

        for i in range(num_qubits):
            circuit.rx(2 * beta_params[rep], i)

    return circuit

In [ ]:
def run_qaoa(graph, num_layers):
  max_cut_paulis = build_max_cut_paulis(graph)

  cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, len(graph))

  circuit = build_qaoa_circuit(cost_hamiltonian, reps=num_layers)

  pm = generate_preset_pass_manager(optimization_level=3)

  candidate_circuit = pm.run(circuit)

  initial_gamma = np.pi
  initial_beta = np.pi/4
  init_params = []
  for rep in range(num_layers):
      init_params.append(initial_gamma)
      init_params.append(initial_beta)

  objective_func_vals = []

  optimizer = L_BFGS_B(maxiter=100)

  bound_cost_func = partial(cost_func_estimator,
                            ansatz=candidate_circuit,
                            hamiltonian=cost_hamiltonian,
                            estimator=estimator,
                            objective_func_vals_list=objective_func_vals)

  result = optimizer.minimize(
      fun=bound_cost_func,
      x0=init_params
  )

  optimized_circuit = candidate_circuit.assign_parameters(result.x)
  optimized_circuit.measure_all()

  pub = (optimized_circuit,)
  job = sampler.run([pub])

  result = job.result()[0]
  counts_bin = result.data.meas.get_counts()

  counts_int = {int(k, 2): v for k, v in counts_bin.items()}
  shots = sum(counts_int.values())
  final_distribution_int = {
      key: val / shots for key, val in counts_int.items()
  }

  best_sol = best_solution(final_distribution_int, cost_hamiltonian)
  best_sol_bitstring = to_bitstring(int(best_sol), len(graph))
  best_sol_bitstring.reverse()

  cut_value = evaluate_sample(best_sol_bitstring, graph)

  result_dist = samples_to_objective_values(
    final_distribution_int, cost_hamiltonian
  )

  return cut_value, best_sol_bitstring, objective_func_vals

# Classic methods



## Brute-Force Solution

In [ ]:
def bruteforce_max_cut(graph: rx.PyGraph, max_nodes_limit: int = 20):
    num_nodes = graph.num_nodes()

    if num_nodes > max_nodes_limit:
        print(f"Число вершин {num_nodes} превышает допустимый предел\n")
        return None, None

    best_cut_value = float('-inf')
    best_partition = None

    for partition_tuple in itertools.product([0, 1], repeat=num_nodes):
        current_partition = list(partition_tuple)
        current_cut_value = evaluate_sample(current_partition, graph)

        if current_cut_value > best_cut_value:
            best_cut_value = current_cut_value
            best_partition = current_partition

    return best_cut_value, best_partition

## Greedy solution

In [ ]:
def greedy_max_cut(graph):
    num_nodes = graph.num_nodes()
    if num_nodes == 0:
        return 0.0, []

    partition = [np.random.randint(0, 2) for _ in range(num_nodes)]
    current_cut_value = evaluate_sample(partition, graph)

    improved = True
    while improved:
        improved = False
        for i in range(num_nodes):
            original_partition_i = partition[i]
            partition[i] = 1 - partition[i]
            new_cut_value = evaluate_sample(partition, graph)

            if new_cut_value > current_cut_value:
                current_cut_value = new_cut_value
                improved = True
            else:
                partition[i] = original_partition_i

    return current_cut_value, partition

# Experiment

## Model Setup

### Noiseless simulator

In [ ]:
clean_backend = AerSimulator(
    method='matrix_product_state',
    mps_log_data=True,
    shots=1024
)

clean_estimator = BackendEstimatorV2(backend=clean_backend,
    options={
        "default_precision": 0.015625,
        "abelian_grouping": True,
        "seed_simulator": 42
    })

clean_sampler = BackendSamplerV2(backend=clean_backend)

### Noisy simulator

In [ ]:
_original_estimator = estimator
_original_sampler = sampler

noise_model = NoiseModel()
error_1 = depolarizing_error(0.005, 1)
noise_model.add_all_qubit_quantum_error(error_1, ['u1', 'u2', 'u3', 'rx', 'ry', 'rz', 'h', 'x', 'y', 'z', 's', 'sdg', 't', 'tdg'])

error_2 = depolarizing_error(0.01, 2)
noise_model.add_all_qubit_quantum_error(error_2, ['cx', 'cz', 'swap', 'rzz'])

readout_error = ReadoutError([[0.99, 0.01],[0.02, 0.98]])
noise_model.add_all_qubit_readout_error(readout_error)

noisy_backend = AerSimulator(
    method='matrix_product_state',
    shots=1024,
    noise_model=noise_model
)

noisy_estimator = BackendEstimatorV2(backend=noisy_backend,
    options={
        "default_precision": 0.015625,
        "abelian_grouping": True,
        "seed_simulator": 42
    })

noisy_sampler = BackendSamplerV2(backend=noisy_backend)

## Graph Generation

In [ ]:
def generate_graph(graph_type: str, num_nodes: int) -> rx.PyGraph:
    graph = rx.PyGraph()
    if graph_type == 'complete':
        graph.add_nodes_from(range(num_nodes))
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):
                random_weight = np.random.uniform(0, 1)
                graph.add_edge(i, j, random_weight)
    elif graph_type == '3_regular':
        if num_nodes % 2 != 0:
            raise ValueError("3-regular graph requires an even number of vertices.")
        if num_nodes < 4:
            raise ValueError("3-regular graph must have at least 4 nodes.")

        nx_graph = nx.random_regular_graph(3, num_nodes, seed=None)

        graph = rx.PyGraph()
        node_indices = [graph.add_node(idx) for idx in range(num_nodes)]
        for u, v in nx_graph.edges():
            random_weight = np.random.uniform(0, 1)
            graph.add_edge(node_indices[u], node_indices[v], random_weight)
    else:
        raise ValueError(f"Unsupported graph type: {graph_type}. Choose from 'complete' or '3_regular'.")

    return graph

In [ ]:
def generate_graphs(vertex_counts_range, graph_types_to_evaluate, num_samples_per_vertex):
    all_generated_graphs_data = []
    for num_nodes in vertex_counts_range:
        for g_type in graph_types_to_evaluate:
            for i in range(num_samples_per_vertex):
                try:
                    graph = generate_graph(g_type, num_nodes)
                    all_generated_graphs_data.append({
                        'graph': graph,
                        'type': g_type,
                        'num_nodes': num_nodes,
                        'sample_id': i + 1
                    })
                except ValueError as e:
                    print(f"    Could not generate {g_type.replace('_', '-')}(N={num_nodes}, sample {i+1}): {e}")
                except Exception as e:
                    print(f"    Error generating {g_type.replace('_', '-')}(N={num_nodes}, sample {i+1}): {e}")

    return all_generated_graphs_data

## Evaluate metrics

Experiment structure:

Parameters:
- Graph types: 3-regular, complete
- Number of vertices: 4 to 16
- Each graph type + number of vertices pair has 20 examples with random weights
- Simulator: Clean

Metrics:
- Approximation ratio (compare to bruteforce)
- Normalized cut (compare to total edge sum)
- Compare to greedy algorithm (>1 means success)
- Compare to QAOA (>1 means success)

In [ ]:
def evaluate_graph_metrics(graph, qaoa_layers, qgoa_layers, bruteforce_max_nodes_limit):
    results = {}
    num_nodes = graph.num_nodes()

    bruteforce_cut, _ = bruteforce_max_cut(graph, max_nodes_limit=bruteforce_max_nodes_limit)
    if bruteforce_cut is None or num_nodes > bruteforce_max_nodes_limit:
        bruteforce_cut = float('nan')

    greedy_cut, _ = greedy_max_cut(graph)
    qaoa_cut, _, _ = run_qaoa(graph, qaoa_layers);
    qgoa_cut, _, _ = run_qgoa(graph, qgoa_layers);

    total_edge_weight = get_total_edge_weight(graph)
    results['normalized_cut'] = qgoa_cut / total_edge_weight
    results['ar'] = qgoa_cut / bruteforce_cut
    results['qgoa_over_greedy'] = qgoa_cut / greedy_cut
    results['qgoa_over_qaoa'] = qgoa_cut / qaoa_cut

    return results

In [ ]:
def run_experiment(system_types_to_evaluate, test_graphs):
    all_results_by_system_type = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for system_type in system_types_to_evaluate:
        if system_type == 'noisy':
            estimator = noisy_estimator
            sampler = noisy_sampler
        else:
            estimator = clean_estimator
            sampler = clean_sampler

        graphs_by_node_type = defaultdict(lambda: defaultdict(list))
        for graph_data in test_graphs:
            graphs_by_node_type[graph_data['num_nodes']][graph_data['type']].append(graph_data)

        for num_nodes in sorted(graphs_by_node_type.keys()):
            for g_type in graph_types_to_evaluate:
                graphs_for_current_type_and_node = graphs_by_node_type[num_nodes][g_type]
                if not graphs_for_current_type_and_node:
                    continue
                for graph_data in graphs_for_current_type_and_node:
                    graph = graph_data['graph']
                    sample_id = graph_data['sample_id']
                    try:
                        start_time = time.time()
                        metrics = evaluate_graph_metrics(graph, qaoa_layers, qgoa_layers, bruteforce_max_nodes_limit)
                        end_time = time.time()
                        elapsed_time = end_time - start_time
                        metrics['time_taken'] = elapsed_time
                        all_results_by_system_type[system_type][g_type][num_nodes].append(metrics)
                    except Exception as e:
                        print("Exception")

    return all_results_by_system_type

In [ ]:
def save_results_to_csv(results_dict, system_name):
    records = []

    for s_type, graph_data_for_system in results_dict.items():
        for g_type, node_data in graph_data_for_system.items():
            for num_nodes, metrics_list in node_data.items():
                for i, metrics in enumerate(metrics_list):
                    record = {
                        'system_type': s_type,
                        'graph_type': g_type,
                        'num_nodes': num_nodes,
                        'sample_id': i + 1,
                        'ar': metrics.get('ar', np.nan),
                        'normalized_cut': metrics.get('normalized_cut', np.nan),
                        'qgoa_over_greedy': metrics.get('qgoa_over_greedy', np.nan),
                        'qgoa_over_qaoa': metrics.get('qgoa_over_qaoa', np.nan),
                        'time_taken': metrics.get('time_taken', np.nan)
                    }
                    records.append(record)

    df_results = pd.DataFrame(records)
    csv_filename = f'experiment_results_{system_name}.csv'
    df_results.to_csv(csv_filename, index=False)
    print(f"Experiment results for {system_name} system saved to {csv_filename}")

In [ ]:
vertex_counts_range = range(4, 17)
num_samples_per_vertex = 20
qaoa_layers = 1
qgoa_layers = 1
bruteforce_max_nodes_limit = 20
graph_types_to_evaluate = ['3_regular', 'complete']

print("Starting experiment...")

for num_nodes in vertex_counts_range:
    graphs = generate_graphs(
        range(num_nodes, num_nodes + 1),
        graph_types_to_evaluate,
        num_samples_per_vertex
    )
    clean_results_for_current_n = run_experiment(['clean'], graphs)

    save_results_to_csv(clean_results_for_current_n, 'clean')

print("\nFinished experiment")

## Time

In [ ]:
for num_nodes_for_graph in range(8, 9):
    try:
        test_graph = generate_3_regular_graph_sample(num_nodes_for_graph)
    except ValueError as e:
        print(f"Error generating complete graph for {num_nodes_for_graph} nodes: {e}")
        continue

    print(f"\nRunning QGOA for a {num_nodes_for_graph}-node complete graph...")
    start_time_qgoa = time.time()
    cut_value_qgoa, best_sol_bitstring_qgoa, objective_func_vals_qgoa = run_qgoa(test_graph, num_layers=1)
    end_time_qgoa = time.time()

    elapsed_time_qgoa = end_time_qgoa - start_time_qgoa

    print(f"QGOA Max Cut Value: {cut_value_qgoa}")
    print(f"Best Solution (QGOA): {best_sol_bitstring_qgoa}")
    print(f"Time taken for QGOA on {num_nodes_for_graph}-node complete graph: {elapsed_time_qgoa:.4f} seconds")

    print(f"\nRunning QAOA for a {num_nodes_for_graph}-node complete graph...")

    start_time_qaoa = time.time()
    cut_value_qaoa, best_sol_bitstring_qaoa, objective_func_vals_qaoa = run_qaoa(test_graph, num_layers=1)
    end_time_qaoa = time.time()

    elapsed_time_qaoa = end_time_qaoa - start_time_qaoa

    print(f"QAOA Max Cut Value: {cut_value_qaoa}")
    print(f"Best Solution (QAOA): {best_sol_bitstring_qaoa}")
    print(f"Time taken for QAOA on {num_nodes_for_graph}-node complete graph: {elapsed_time_qaoa:.4f} seconds")

    print(f"\nTime comparison: QGOA took {elapsed_time_qgoa:.4f}s, QAOA took {elapsed_time_qaoa:.4f}s.")

    plt.figure(figsize=(10, 6))
    plt.plot(objective_func_vals_qgoa, label='QGOA Convergence')
    plt.plot(objective_func_vals_qaoa, label='QAOA Convergence')
    plt.xlabel('Optimization Iteration')
    plt.ylabel('Objective Function Value')
    plt.title(f'Convergence Rate of QGOA vs QAOA for {num_nodes_for_graph}-node complete graph')
    plt.legend()
    plt.grid(True)
    plt.show()
    print("-" * 30)
